            # Run DeepMD inference against a reference trajectory

            This notebook evaluates a frozen DeepMD model on an ASE trajectory that already contains
            reference energies and forces, then writes plain-text comparison files for downstream analysis.

            ## Outputs
            - `energies_forces.txt`: per-frame DFT vs DeepMD energies
            - `energies_forces_atoms.txt`: per-atom DFT vs DeepMD force components

            ## Before you run it
            Run this notebook from the DeepMD Python environment.
            Place one or more labelled validation/test trajectories in `model_analysis/input/`.
            


In [ ]:
            from pathlib import Path

            import numpy as np
            from ase.io import read
            from deepmd.infer import DeepPot
            from tqdm import tqdm

            try:
                from ase.calculators.deepmd import DeepMDCalculator as DP
            except Exception:
                from deepmd.calculator import DP
            


            ## Configuration

            Select the model and reference trajectory here. Relative paths are resolved from the repository root.
            


In [ ]:
            PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "model_analysis" else Path.cwd().resolve()

            MODEL_FILE = PROJECT_ROOT / "model_analysis" / "model" / "graph.pb"
            INPUT_DIR = PROJECT_ROOT / "model_analysis" / "input"
            TRAJECTORY_PATTERN = "*.traj"
            TRAJECTORY_INDEX = 0
            OUTPUT_DIR = PROJECT_ROOT / "model_analysis" / "outputs" / "inference"

            INPUT_DIR.mkdir(parents=True, exist_ok=True)
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

            print(f"Project root: {PROJECT_ROOT}")
            print(f"Model file: {MODEL_FILE}")
            print(f"Input directory: {INPUT_DIR}")
            print(f"Output directory: {OUTPUT_DIR}")
            


In [ ]:
            if not MODEL_FILE.exists():
                raise FileNotFoundError(
                    f"Model file not found: {MODEL_FILE}\n"
                    "Update MODEL_FILE in the configuration cell before running the notebook."
                )

            trajectory_files = sorted(INPUT_DIR.glob(TRAJECTORY_PATTERN))
            if not trajectory_files:
                raise FileNotFoundError(
                    f"No trajectory file matched '{TRAJECTORY_PATTERN}' in {INPUT_DIR}\n"
                    "Add one or more `.traj` files to model_analysis/input/ or update TRAJECTORY_PATTERN."
                )

            if TRAJECTORY_INDEX >= len(trajectory_files):
                raise IndexError(
                    f"TRAJECTORY_INDEX={TRAJECTORY_INDEX} but only {len(trajectory_files)} matching file(s) were found."
                )

            print("Available trajectory files:")
            for idx, path in enumerate(trajectory_files):
                print(f"  [{idx}] {path.name}")

            trajectory_file = trajectory_files[TRAJECTORY_INDEX]
            print(f"Selected trajectory: {trajectory_file.name}")

            frames = read(str(trajectory_file), ":")
            print(f"Loaded {len(frames)} frames from {trajectory_file}")
            


            ## Load the model and attach the DeepMD calculator
            


In [ ]:
            model = DeepPot(str(MODEL_FILE))
            type_map = model.get_type_map()

            print(f"type_map: {type_map}")

            calculator = DP(model=str(MODEL_FILE), type_map=type_map)
            


In [ ]:
            reference_frames = []
            predicted_frames = []

            for atoms in frames:
                atoms_ref = atoms.copy()
                atoms_ref.calc = atoms.calc
                reference_frames.append(atoms_ref)

                atoms_dp = atoms.copy()
                atoms_dp.calc = calculator
                predicted_frames.append(atoms_dp)

            print(f"Prepared {len(reference_frames)} reference/prediction frame pairs")
            


            ## Evaluate and write comparison files
            


In [ ]:
            frame_output = OUTPUT_DIR / "energies_forces.txt"
            atom_output = OUTPUT_DIR / "energies_forces_atoms.txt"

            with frame_output.open("w", encoding="utf-8") as fh_energy, atom_output.open("w", encoding="utf-8") as fh_atom:
                fh_energy.write("# frame energy_DFT_eV energy_DP_eV\n")
                fh_atom.write(
                    "# frame atom element fx_DFT fy_DFT fz_DFT fx_DP fy_DP fz_DP energy_DFT energy_DP\n"
                )

                for frame_index, (atoms_ref, atoms_dp) in enumerate(
                    tqdm(
                        zip(reference_frames, predicted_frames),
                        total=len(reference_frames),
                        desc="Evaluating",
                    )
                ):
                    energy_ref = atoms_ref.get_potential_energy()
                    forces_ref = atoms_ref.get_forces()

                    energy_dp = atoms_dp.get_potential_energy()
                    forces_dp = atoms_dp.get_forces()

                    fh_energy.write(f"{frame_index:6d} {energy_ref: .10f} {energy_dp: .10f}\n")

                    for atom_index, symbol in enumerate(atoms_ref.get_chemical_symbols()):
                        fxr, fyr, fzr = forces_ref[atom_index]
                        fxd, fyd, fzd = forces_dp[atom_index]
                        fh_atom.write(
                            f"{frame_index:6d} {atom_index:6d} {symbol:2s} "
                            f"{fxr: .6f} {fyr: .6f} {fzr: .6f} "
                            f"{fxd: .6f} {fyd: .6f} {fzd: .6f} "
                            f"{energy_ref: .10f} {energy_dp: .10f}\n"
                        )

            print(f"Wrote frame-level comparison file to: {frame_output}")
            print(f"Wrote atom-level comparison file to: {atom_output}")
            


            ## Next step

            Open `dp_correlation_plots.ipynb` and point it at the files written in `model_analysis/outputs/inference/`.
            
